# M³ fast: 20 PANGAEA + 20 iNaturalist datasets

This notebook runs the **two-agent, one-revision-round** M³ path with SURF `default-text-large`. It uses a saved, seeded 20-file manifest per corpus and compares the *same source files* with the existing `default-text-large` ReAct and single-LLM outputs. Each successful M³ run retains its JSONL conversation/tool trace.

**PANGAEA target:** the seven shared descriptive and technical fields: `name`, `description`, `keywords`, `spatialCoverage`, `temporalCoverage`, `spatial`, and `temporal`. M³ uses `pangaea_descriptive_technical_standard`; the two baseline outputs use the same seven field definitions under `single_llm_pangaea_standard`. **iNaturalist target:** `spatialCoverage` and `temporalCoverage`.

**Scoring (following the companion `process_data_metadata_agent` notebooks):** PANGAEA descriptive faithfulness (RubricsScore 1–5, normalized to 0–1) and reference similarity (FactualCorrectness precision, 0–1), plus technical extent accuracy, presence precision, and presence recall. iNaturalist reports exact technical extent accuracy only. Technical accuracy is the mean of six field-level match rates over all 20 selected files. Missing/failed predictions score zero for end-to-end accuracy; a judge failure stays unscored and blocks the final descriptive table. Precision and recall describe whether a bound was supplied when the reference has one; they **do not** measure whether its value was correct. PANGAEA dates match by calendar day and coordinates by `numpy.isclose`; iNaturalist dates and coordinates match exactly, as in the companion evaluator.

The single-LLM baseline records context-window skips for all current iNaturalist CSVs. This notebook preserves those 20 skips and reports their end-to-end accuracy as zero, with a separate success count. The descriptive reference is incomplete creator metadata, so similarity measures agreement with that reference, not factual validity. The judge uses the companion notebook's Devstral model, separately from the `default-text-large` extraction model.

**Run order:** execute cells from top to bottom. The M³ and judge cells checkpoint after every dataset; rerunning resumes successes. Set `M3_MAX_NEW_RUNS=0` in the environment to inspect the manifest and existing baseline results without new model calls.

In [1]:
import hashlib
import json
import os
import random
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src' / 'config.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the metadata_agent checkout')
ROOT = ROOT.resolve()
EVAL_ROOT = ROOT.parent / 'process_data_metadata_agent'
if not (EVAL_ROOT / 'notebooks' / '09-TvdP-evaluate-descriptive-metadata.ipynb').is_file():
    raise FileNotFoundError(f'Companion evaluation checkout not found: {EVAL_ROOT}')
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')
os.environ['LLM_PROVIDER'] = 'surf'

from src.orchestrator.orchestrator import Orchestrator
from src.standards import METADATA_STANDARDS, get_schema_for_standard

MODEL = 'default-text-large'
TOPOLOGY = 'fast'
SEED = 9
N_PER_CORPUS = 20
MAX_NEW_RUNS = int(os.getenv('M3_MAX_NEW_RUNS', '-1'))  # -1 runs all pending files
PANGAEA_STANDARD = 'pangaea_descriptive_technical_standard'
INAT_STANDARD = 'croissant_inaturalist_standard'
PANGAEA_FIELDS = ('name', 'description', 'keywords', 'spatialCoverage',
                  'temporalCoverage', 'spatial', 'temporal')
INAT_FIELDS = ('spatialCoverage', 'temporalCoverage')
DATA_DIRS = {
    'pangaea': ROOT / 'data/experiment/pangaea_datasets',
    'inaturalist': ROOT / 'data/experiment/inaturalist_100_species_1000_obs/inaturalist_100_species_1000_obs',
}
ANNOTATIONS = EVAL_ROOT / 'outputs/pangaea/sample_20260629_163144_filtered_top5_per_topic.csv'
EXTENT_CACHE = EVAL_ROOT / 'outputs/pangaea/sample_20260629_163144_extents.json'
DESCRIPTIVE_CACHE = EVAL_ROOT / 'outputs/pangaea/df_scores_pangaea_descriptive_with_baselines.csv'
OUT = ROOT / 'outputs/m3_fast_20_default-text-large'
OUT.mkdir(parents=True, exist_ok=True)

for corpus, data_dir in DATA_DIRS.items():
    if not data_dir.is_dir():
        raise FileNotFoundError(data_dir)
if MODEL != 'default-text-large' or TOPOLOGY != 'fast':
    raise ValueError('This notebook is pinned to the requested model and topology')
assert len(get_schema_for_standard(PANGAEA_STANDARD).model_fields) == 7
assert len(get_schema_for_standard(INAT_STANDARD).model_fields) == 2
print('Project:', ROOT)
print('Evaluation sources:', EVAL_ROOT)
print('Output:', OUT)
print('New M³ runs allowed this session:', 'all pending' if MAX_NEW_RUNS < 0 else MAX_NEW_RUNS)
print('SURF credentials present:', bool(os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')))


Project: /home/com3dian/Github/metadata_agent
Evaluation sources: /home/com3dian/Github/process_data_metadata_agent
Output: /home/com3dian/Github/metadata_agent/outputs/m3_fast_20_default-text-large
New M³ runs allowed this session: all pending
SURF credentials present: True


In [2]:
def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json_atomic(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(payload, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
    temporary.replace(path)


manifest_path = OUT / 'manifest.json'
annotations = pd.read_csv(ANNOTATIONS)
if len(annotations) != 75 or annotations['id'].duplicated().any():
    raise ValueError('Expected 75 unique PANGAEA reference rows')
if len(list(DATA_DIRS['inaturalist'].glob('*.csv'))) != 100:
    raise ValueError('Expected 100 iNaturalist CSV files')

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('seed') != SEED or manifest.get('n_per_corpus') != N_PER_CORPUS:
        raise ValueError('Saved manifest uses different selection settings')
else:
    # One file from each of 15 PANGAEA topics, plus five seeded extra files.
    first = annotations.groupby('topic', sort=True).sample(n=1, random_state=SEED)
    extra = annotations.loc[~annotations['id'].isin(first['id'])].sample(
        n=N_PER_CORPUS-len(first), random_state=SEED
    )
    pangaea_ids = sorted(str(int(value)) for value in pd.concat([first, extra])['id'])
    inat_ids = sorted(random.Random(SEED).sample(
        [path.stem for path in sorted(DATA_DIRS['inaturalist'].glob('*.csv'))],
        N_PER_CORPUS,
    ))
    manifest = {'seed': SEED, 'n_per_corpus': N_PER_CORPUS, 'model': MODEL,
                'topology': TOPOLOGY, 'datasets': {
                    'pangaea': [{'id': stem,
                                 'source_sha256': sha256_file(DATA_DIRS['pangaea'] / f'{stem}.csv')}
                                for stem in pangaea_ids],
                    'inaturalist': [{'id': stem,
                                    'source_sha256': sha256_file(DATA_DIRS['inaturalist'] / f'{stem}.csv')}
                                   for stem in inat_ids],
                }}
    write_json_atomic(manifest_path, manifest)

for corpus in DATA_DIRS:
    selected = manifest['datasets'][corpus]
    if len(selected) != N_PER_CORPUS or len({x['id'] for x in selected}) != N_PER_CORPUS:
        raise ValueError(f'Invalid saved {corpus} selection')
    for item in selected:
        source = DATA_DIRS[corpus] / f"{item['id']}.csv"
        if not source.is_file() or sha256_file(source) != item['source_sha256']:
            raise ValueError(f'Source changed since manifest creation: {source}')
    print(corpus, len(selected), 'datasets:', ', '.join(x['id'] for x in selected))
print('Manifest:', manifest_path)


pangaea 20 datasets: 899705, 913868, 917831, 917841, 944025, 953747, 958774, 959885, 961119, 961147, 963863, 965135, 965137, 971851, 972178, 973035, 973041, 982395, 987056, 990054
inaturalist 20 datasets: acianthus_fornicatus, amyema_preissii, asemosyrphus_mexicanus, chlorurus_bleekeri, cinnyris_pulchellus, clarias_gariepinus, eucalyptus_regnans, hibbertia_hypericoides, hymeniacidon_perlevis, lachnagrostis_billardierei, lactuca_graminifolia, muhlenbergia_lindheimeri, nannophya_pygmaea, pectocarya_recurvata, pterostylis_graminea, ptychoramphus_aleuticus, sapium_haematospermum, sesamum_indicum, sphenophorus_interstitialis, struthiolaria_papulosa
Manifest: /home/com3dian/Github/metadata_agent/outputs/m3_fast_20_default-text-large/manifest.json


## 1. Run M³ fast

Every success and failure is saved immediately. Successful files with the same source hash, model, topology, standard, and expected fields are reused. Failures remain retryable. The model/tool conversation is in `trace_path`; no credentials are written to result JSON. A run may take a long time because each plan step has two agents and one critique/revision cycle.

In [3]:
import logging

# Stream orchestrator/player logs into this cell so the current step and agent
# are visible while a dataset is running (same setup as the other experiment notebooks).
log = logging.getLogger()
log.setLevel(logging.INFO)
for handler in log.handlers[:]:
    log.removeHandler(handler)
handler = logging.StreamHandler(sys.stdout)
handler.setLevel(logging.INFO)
handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(name)s - %(message)s'))
log.addHandler(handler)


def m3_result_path(corpus: str, stem: str) -> Path:
    return OUT / 'm3' / corpus / f'{stem}.json'


def valid_m3_success(path: Path, item: dict, corpus: str) -> bool:
    if not path.is_file():
        return False
    try:
        record = json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return False
    standard = PANGAEA_STANDARD if corpus == 'pangaea' else INAT_STANDARD
    fields = PANGAEA_FIELDS if corpus == 'pangaea' else INAT_FIELDS
    return (record.get('status') == 'ok'
            and record.get('model') == MODEL
            and record.get('topology') == TOPOLOGY
            and record.get('standard') == standard
            and record.get('source_sha256') == item['source_sha256']
            and isinstance(record.get('metadata'), dict)
            and set(fields).issubset(record['metadata']))


def run_m3_one(corpus: str, item: dict) -> dict:
    source = DATA_DIRS[corpus] / f"{item['id']}.csv"
    standard = PANGAEA_STANDARD if corpus == 'pangaea' else INAT_STANDARD
    fields = PANGAEA_FIELDS if corpus == 'pangaea' else INAT_FIELDS
    base = {'dataset_id': item['id'], 'csv': str(source), 'source_sha256': item['source_sha256'],
            'method': 'M3_fast', 'provider': 'surf', 'model': MODEL,
            'topology': TOPOLOGY, 'standard': standard,
            'finished_utc': datetime.now(timezone.utc).isoformat()}
    started = time.monotonic()
    orchestrator = None
    try:
        orchestrator = Orchestrator(topology_name=TOPOLOGY, model_name=MODEL, provider='surf')
        result = orchestrator.run(
            source=str(source), metadata_standard=METADATA_STANDARDS[standard],
            metadata_standard_name=standard, name=item['id'],
        )
        metadata = result.final_metadata if result is not None else None
        if result is None or not result.success:
            raise RuntimeError((result.error if result is not None else 'Planning failed') or 'Run failed')
        if not isinstance(metadata, dict) or not set(fields).issubset(metadata):
            raise ValueError(f'Missing requested metadata fields: {sorted(set(fields)-set(metadata or {}))}')
        metadata = get_schema_for_standard(standard).model_validate(metadata).model_dump(
            mode='json', by_alias=True
        )
        return {**base, 'status': 'ok', 'metadata': metadata,
                'plan_steps_count': result.plan_steps_count,
                'steps_completed': result.steps_completed,
                'trace_path': result.trace_path,
                'runtime_seconds': round(time.monotonic()-started, 2)}
    except Exception as exc:
        return {**base, 'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
                'trace_path': getattr(orchestrator, 'last_trace_path', None),
                'runtime_seconds': round(time.monotonic()-started, 2)}


if MAX_NEW_RUNS != 0 and not (os.getenv('SURF_API_KEY') and os.getenv('SURF_API_BASE')):
    raise RuntimeError('SURF_API_KEY and SURF_API_BASE are required for new M³ runs')

new_runs = 0
for corpus in ('pangaea', 'inaturalist'):
    selected = manifest['datasets'][corpus]
    for item in tqdm(selected, desc=f'M³ fast {corpus}'):
        path = m3_result_path(corpus, item['id'])
        if valid_m3_success(path, item, corpus):
            continue
        if 0 <= MAX_NEW_RUNS <= new_runs:
            continue
        print(f'Starting M³ {corpus} {item["id"]}', flush=True)
        record = run_m3_one(corpus, item)
        write_json_atomic(path, record)
        new_runs += 1
        print(corpus, item['id'], record['status'],
              record.get('error', ''), 'trace:', record.get('trace_path'))
print('New M³ attempts:', new_runs)
for corpus in ('pangaea', 'inaturalist'):
    print(corpus, 'saved successes:', sum(valid_m3_success(m3_result_path(corpus, x['id']), x, corpus)
                                          for x in manifest['datasets'][corpus]), '/', N_PER_CORPUS)


M³ fast pangaea:   0%|          | 0/20 [00:00<?, ?it/s]

Starting M³ pangaea 899705
2026-10-03 14:49:03,065 - INFO - root - PlanExecutor initialized with topology: fast
2026-10-03 14:49:03,065 - INFO - root -   Players per step: 2
2026-10-03 14:49:03,066 - INFO - root -   Debate rounds: 1
2026-10-03 14:49:03,066 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-10-03 14:49:03,066 - INFO - root - Orchestrator initialized with topology: fast
2026-10-03 14:49:03,067 - INFO - root - Run trace: /home/com3dian/Github/metadata_agent/outputs/traces/899705_06943760509e447e8130c1c403ef9939.jsonl
2026-10-03 14:49:03,068 - INFO - root - ============================================================
2026-10-03 14:49:03,068 - INFO - root - STARTING ORCHESTRATION
2026-10-03 14:49:03,068 - INFO - root - Context: 899705
2026-10-03 14:49:03,068 - INFO - root - Type: single_csv
2026-10-03 14:49:03,068 - INFO - root - Resources: ['

M³ fast inaturalist:   0%|          | 0/20 [00:00<?, ?it/s]

Starting M³ inaturalist acianthus_fornicatus
2026-10-03 15:56:32,369 - INFO - root - PlanExecutor initialized with topology: fast
2026-10-03 15:56:32,370 - INFO - root -   Players per step: 2
2026-10-03 15:56:32,370 - INFO - root -   Debate rounds: 1
2026-10-03 15:56:32,371 - INFO - root -   Player pool: ['dataset_schema_preview', 'data_analyst', 'schema_expert', 'metadata_specialist', 'relationship_analyst', 'spatial_temporal_specialist', 'critic']
2026-10-03 15:56:32,371 - INFO - root - Orchestrator initialized with topology: fast
2026-10-03 15:56:32,371 - INFO - root - Run trace: /home/com3dian/Github/metadata_agent/outputs/traces/acianthus_fornicatus_79cd9d087db04e24978647f7e7d67727.jsonl
2026-10-03 15:56:32,372 - INFO - root - ============================================================
2026-10-03 15:56:32,372 - INFO - root - STARTING ORCHESTRATION
2026-10-03 15:56:32,372 - INFO - root - Context: acianthus_fornicatus
2026-10-03 15:56:32,373 - INFO - root - Type: single_csv
2026-10

## 2. Load the three methods on the exact saved manifest

Baseline records must match the chosen CSV hash, model, and expected standard. Missing and skipped files remain explicit rows. Existing ReAct and single-LLM outputs are read only; this notebook does not rerun them.

In [4]:
def baseline_record(corpus: str, method: str, item: dict) -> dict:
    suffix = 'react' if method == 'ReAct' else 'single_llm'
    folder = ROOT / 'outputs' / f'{suffix}_{corpus}_default-text-large'
    success = folder / f"{item['id']}_{suffix}.json"
    skip = folder / f"{item['id']}_{suffix}.skipped.json"
    path = success if success.is_file() else skip
    if not path.is_file():
        return {'status': 'missing', 'metadata': None, 'path': None, 'reason': 'no saved result'}
    record = json.loads(path.read_text(encoding='utf-8'))
    expected_standard = ('single_llm_pangaea_standard' if corpus == 'pangaea'
                         else INAT_STANDARD)
    if (record.get('model') != MODEL or record.get('standard') != expected_standard
            or record.get('source_sha256') != item['source_sha256']):
        raise ValueError(f'Baseline provenance mismatch: {path}')
    status = record.get('status')
    if status not in {'ok', 'skipped'}:
        raise ValueError(f'Unexpected baseline status in {path}: {status}')
    fields = PANGAEA_FIELDS if corpus == 'pangaea' else INAT_FIELDS
    metadata = record.get('metadata') if status == 'ok' else None
    if status == 'ok' and (not isinstance(metadata, dict) or not set(fields).issubset(metadata)):
        raise ValueError(f'Incomplete baseline metadata: {path}')
    return {'status': status, 'metadata': metadata, 'path': str(path),
            'reason': record.get('reason')}


rows = []
for corpus in ('pangaea', 'inaturalist'):
    for item in manifest['datasets'][corpus]:
        for method in ('M3_fast', 'ReAct', 'SingleLLM'):
            if method == 'M3_fast':
                path = m3_result_path(corpus, item['id'])
                record = (json.loads(path.read_text(encoding='utf-8')) if path.is_file()
                          else {'status': 'missing', 'error': 'not run'})
                if record.get('status') == 'ok' and not valid_m3_success(path, item, corpus):
                    raise ValueError(f'Stale M³ result: {path}')
                loaded = {'status': record['status'], 'metadata': record.get('metadata'),
                          'path': str(path) if path.is_file() else None,
                          'reason': record.get('error')}
            else:
                loaded = baseline_record(corpus, method, item)
            rows.append({'corpus': corpus, 'dataset_id': item['id'], 'method': method,
                         'source_sha256': item['source_sha256'], **loaded})

predictions = pd.DataFrame(rows)
assert predictions.groupby(['corpus', 'method']).size().eq(N_PER_CORPUS).all()
print(predictions.groupby(['corpus', 'method', 'status']).size().rename('n').to_string())
print('Sources with statuses are ready for evaluation.')


corpus       method     status 
inaturalist  M3_fast    ok         20
             ReAct      ok         20
             SingleLLM  skipped    20
pangaea      M3_fast    failed      1
                        ok         19
             ReAct      ok         20
             SingleLLM  ok         19
                        skipped     1
Sources with statuses are ready for evaluation.


## 3. Technical reference and companion-style scoring

PANGAEA uses the companion evaluator's cached creator extents, including absent extents. iNaturalist reference bounds are calculated from all rows of each selected CSV. This section makes no model calls.

In [5]:
SPATIAL = ('min_lon', 'max_lon', 'min_lat', 'max_lat')
TEMPORAL = ('min_time', 'max_time')
TECH_FIELDS = SPATIAL + TEMPORAL


def parsed_time(value):
    if value is None or not isinstance(value, str) or not value.strip():
        return None
    text = value.strip().removesuffix('Z')
    if len(text) == 10:
        text += 'T00:00:00'
    elif len(text) == 16 and text[10] == 'T':
        text += ':00'
    parsed = pd.to_datetime(text[:19], errors='coerce')
    return None if pd.isna(parsed) else parsed.to_pydatetime().replace(tzinfo=None)


def numeric(value):
    try:
        number = float(value)
        return number if np.isfinite(number) else None
    except (TypeError, ValueError):
        return None


def coverage_values(metadata):
    spatial = metadata.get('spatialCoverage') if isinstance(metadata, dict) else None
    temporal = metadata.get('temporalCoverage') if isinstance(metadata, dict) else None
    spatial = spatial if isinstance(spatial, dict) else {}
    temporal = temporal if isinstance(temporal, dict) else {}
    return {**{field: numeric(spatial.get(field)) for field in SPATIAL},
            'min_time': parsed_time(temporal.get('from')),
            'max_time': parsed_time(temporal.get('to'))}


extent_cache = json.loads(EXTENT_CACHE.read_text(encoding='utf-8'))
if len(extent_cache) != 75:
    raise ValueError('PANGAEA extent cache should cover all 75 annotated datasets')


def pangaea_reference(stem):
    entry = extent_cache[stem]
    box = entry.get('spat_extent') or {}
    times = entry.get('temp_extent') or [None, None]
    return {'min_lon': numeric(box.get('westBoundLongitude')),
            'max_lon': numeric(box.get('eastBoundLongitude')),
            'min_lat': numeric(box.get('southBoundLatitude')),
            'max_lat': numeric(box.get('northBoundLatitude')),
            'min_time': parsed_time(times[0]), 'max_time': parsed_time(times[1])}


def inat_reference(stem):
    data = pd.read_csv(DATA_DIRS['inaturalist'] / f'{stem}.csv')
    for field in ('decimalLatitude', 'decimalLongitude', 'eventDate'):
        if field not in data:
            raise ValueError(f'{stem} lacks {field}')
    dates = data['eventDate'].map(parsed_time)
    if dates.isna().any():
        raise ValueError(f'{stem} has unparseable eventDate rows')
    lat = pd.to_numeric(data['decimalLatitude'], errors='coerce')
    lon = pd.to_numeric(data['decimalLongitude'], errors='coerce')
    return {'min_lon': numeric(lon.min()), 'max_lon': numeric(lon.max()),
            'min_lat': numeric(lat.min()), 'max_lat': numeric(lat.max()),
            'min_time': dates.min(), 'max_time': dates.max()}


references = {
    'pangaea': {x['id']: pangaea_reference(x['id']) for x in manifest['datasets']['pangaea']},
    'inaturalist': {x['id']: inat_reference(x['id']) for x in manifest['datasets']['inaturalist']},
}


def value_match(corpus, field, reference, prediction):
    if reference is None or prediction is None:
        return False
    if field in SPATIAL:
        return (bool(np.isclose(reference, prediction)) if corpus == 'pangaea'
                else reference == prediction)
    return (reference.date() == prediction.date() if corpus == 'pangaea'
            else reference == prediction)


technical_rows = []
for row in predictions.itertuples(index=False):
    reference = references[row.corpus][row.dataset_id]
    predicted = coverage_values(row.metadata if row.status == 'ok' else None)
    for field in TECH_FIELDS:
        truth, value = reference[field], predicted[field]
        technical_rows.append({
            'corpus': row.corpus, 'dataset_id': row.dataset_id, 'method': row.method,
            'field': field, 'reference_present': truth is not None,
            'prediction_present': value is not None,
            'correct': value_match(row.corpus, field, truth, value),
            'reference': truth, 'prediction': value,
        })
technical = pd.DataFrame(technical_rows)
assert len(technical) == 2 * 3 * N_PER_CORPUS * len(TECH_FIELDS)
technical.to_csv(OUT / 'technical_per_field.csv', index=False)
print('Saved field-level technical comparison:', OUT / 'technical_per_field.csv')


Saved field-level technical comparison: /home/com3dian/Github/metadata_agent/outputs/m3_fast_20_default-text-large/technical_per_field.csv


## 4. PANGAEA descriptive judge

The two baseline faithfulness/similarity columns are reused from the companion evaluator's saved scores for the same file IDs and model. New M³ successes are scored by the *same* Devstral RubricsScore and FactualCorrectness setup, with the same schema, first five rows, row count, and reference description. Each M³ judge result is checkpointed. Install `ragas`, `langchain-openai`, and `tabulate` in this kernel if this cell reports an import error. Set `M3_MAX_NEW_RUNS=0` to skip extraction while developing the evaluation cells.

In [6]:
from langchain_openai import ChatOpenAI
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import FactualCorrectness, RubricsScore

JUDGE_MODEL = 'mistralai/Devstral-Small-2-24B-Instruct-2512'
SURF_API_KEY = os.getenv('SURF_API_KEY')
SURF_API_BASE = os.getenv('SURF_API_BASE') or 'https://willma.surf.nl/api/v0/'
if not SURF_API_KEY:
    raise RuntimeError('SURF_API_KEY is required for the descriptive judge')

judge_llm = LangchainLLMWrapper(ChatOpenAI(
    model=JUDGE_MODEL, api_key=SURF_API_KEY, base_url=SURF_API_BASE,
    default_headers={'X-API-KEY': SURF_API_KEY}, temperature=0,
    max_tokens=2048, timeout=600, max_retries=4,
))
grounding_scorer = RubricsScore(rubrics={
    'score1_description': 'Makes claims that contradict the data or reference fields/values that do not exist in it.',
    'score2_description': 'Several claims are unsupported by the data or appear invented.',
    'score3_description': 'Mostly supportable, but contains at least one unsupported or speculative claim.',
    'score4_description': 'Every claim is supportable from the data; some claims are vague or generic.',
    'score5_description': 'Every claim is directly supported by the data and is specific to it.',
}, llm=judge_llm)
similarity_scorer = FactualCorrectness(llm=judge_llm, mode='precision', atomicity='low')


def serialize_dataset(frame):
    schema = '\n'.join(f'- {column} ({dtype})' for column, dtype in frame.dtypes.items())
    sample = frame.head(5).to_markdown(index=False)
    return f'Columns:\n{schema}\n\nSample rows:\n{sample}\n\nRow count: {len(frame)}'


async def score_m3_description(stem, metadata, reference_description):
    data = pd.read_csv(DATA_DIRS['pangaea'] / f'{stem}.csv')
    context = serialize_dataset(data)
    predicted = {key: metadata.get(key) for key in ('name', 'description', 'keywords')}
    predicted['keywords'] = str(predicted['keywords'])
    if any(not isinstance(predicted[key], str) or not predicted[key].strip() for key in predicted):
        raise ValueError(f'Missing descriptive field in M³ output {stem}')
    abstract = '\n\n'.join(f'{key}: {predicted[key]}' for key in predicted)
    grounding_sample = SingleTurnSample(
        user_input=(
            'A model generated metadata for a dataset from the data alone.\n\n'
            f'DATASET:\n{context}\n\n'
            'Judge only whether the TITLE, DESCRIPTION, and KEYWORDS below are grounded in this data — '
            'whether its claims could be inferred from the data. Do not reward or '
            'penalize style, completeness, or omissions.'
        ),
        response=abstract,
    )
    similarity_sample = SingleTurnSample(
        response=predicted['description'], reference=reference_description,
    )
    rating = await grounding_scorer.single_turn_ascore(grounding_sample)
    similarity = await similarity_scorer.single_turn_ascore(similarity_sample)
    return (float(rating)-1)/4, float(similarity)


cached = pd.read_csv(DESCRIPTIVE_CACHE).set_index('id')
selected_pangaea = [int(x['id']) for x in manifest['datasets']['pangaea']]
if not set(selected_pangaea).issubset(cached.index):
    raise ValueError('Selected PANGAEA IDs are missing from the descriptive score table')
for method in ('ReAct', 'SingleLLM'):
    for prefix in ('faith', 'sim'):
        key = f'{prefix}_{method}_{MODEL}'
        if key not in cached or cached.loc[selected_pangaea, key].isna().any():
            raise ValueError(f'Incomplete cached baseline scores: {key}')

judge_dir = OUT / 'descriptive_judge'
judge_dir.mkdir(parents=True, exist_ok=True)
annotation_by_id = annotations.set_index('id')


async def score_pending_m3():
    for item in tqdm(manifest['datasets']['pangaea'], desc='M³ descriptive judge'):
        stem = item['id']
        prediction = predictions.loc[
            (predictions.corpus == 'pangaea') &
            (predictions.dataset_id == stem) &
            (predictions.method == 'M3_fast')
        ].iloc[0]
        if prediction['status'] != 'ok':
            continue  # Missing extraction output is zero in the final end-to-end table.
        output_path = judge_dir / f'{stem}.json'
        if output_path.is_file():
            saved = json.loads(output_path.read_text(encoding='utf-8'))
            if (saved.get('status') == 'ok' and saved.get('source_sha256') == item['source_sha256']
                    and saved.get('metadata_sha256') == hashlib.sha256(json.dumps(
                        prediction['metadata'], sort_keys=True).encode()).hexdigest()):
                continue
        try:
            last_error = None
            for attempt in range(1, 4):
                try:
                    faith, similarity = await score_m3_description(
                        stem, prediction['metadata'], annotation_by_id.loc[int(stem), 'description']
                    )
                    last_error = None
                    break
                except Exception as exc:
                    last_error = exc
                    if type(exc).__name__ not in {'APITimeoutError', 'APIConnectionError', 'RateLimitError'}:
                        raise
                    print(f'judge {stem} attempt {attempt} failed: {type(exc).__name__}: {exc}')
            if last_error is not None:
                raise last_error
            if not (0 <= faith <= 1 and 0 <= similarity <= 1):
                raise ValueError(f'Judge score outside 0-1: {faith}, {similarity}')
            record = {'dataset_id': stem, 'status': 'ok', 'source_sha256': item['source_sha256'],
                      'metadata_sha256': hashlib.sha256(json.dumps(
                          prediction['metadata'], sort_keys=True).encode()).hexdigest(),
                      'judge_model': JUDGE_MODEL, 'faithfulness': faith,
                      'similarity': similarity}
        except Exception as exc:
            record = {'dataset_id': stem, 'status': 'judge_error',
                      'error': f'{type(exc).__name__}: {exc}'}
        write_json_atomic(output_path, record)
        print('judge', stem, record['status'], record.get('error', ''))


await score_pending_m3()
print('Judge checkpoint folder:', judge_dir)


/home/com3dian/miniconda3/envs/sam2/lib/python3.12/site-packages/instructor/providers/gemini/client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # type: ignore[import-not-found]
/tmp/ipykernel_98383/2077210797.py:4: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import FactualCorrectness, RubricsScore
/tmp/ipykernel_98383/2077210797.py:4: DeprecationWarning: Importing RubricsScore from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections'

M³ descriptive judge:   0%|          | 0/20 [00:00<?, ?it/s]

2026-10-03 16:29:05,055 - INFO - openai._base_client - Retrying request to /chat/completions in 0.479950 seconds
judge 899705 judge_error APITimeoutError: Request timed out.
2026-10-03 16:31:29,117 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-03 16:31:33,442 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-03 16:31:45,410 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
judge 913868 ok 
2026-10-03 16:31:48,411 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-03 16:31:53,186 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
2026-10-03 16:32:07,558 - INFO - httpx - HTTP Request: POST https://willma.surf.nl/api/v0/chat/completions "HTTP/1.1 200 OK"
judge 917831 ok 
2026-10-03 16:32:09,986 - INFO - httpx - H

## 5. Paired summary and saved comparison

End-to-end means all 20 selected files stay in the denominator. Technical accuracy counts correct values; technical precision/recall count presence against the reference, matching notebook 07. Scores are presented as percentages. The table includes success/skip/failure counts to show coverage. No ranking is reported if any successful M³ descriptive output still has a judge error.

In [9]:
descriptive_rows = []
for row in predictions.loc[predictions.corpus == 'pangaea'].itertuples(index=False):
    if row.method == 'M3_fast':
        if row.status != 'ok':
            faith, similarity = 0.0, 0.0
        else:
            score_path = judge_dir / f'{row.dataset_id}.json'
            score = json.loads(score_path.read_text()) if score_path.is_file() else {}
            if score.get('status') != 'ok':
                raise RuntimeError(f'M³ descriptive judge missing/failed for {row.dataset_id}: {score_path}')
            faith, similarity = score['faithfulness'], score['similarity']
    else:
        faith = float(cached.loc[int(row.dataset_id), f'faith_{row.method}_{MODEL}'])
        similarity = float(cached.loc[int(row.dataset_id), f'sim_{row.method}_{MODEL}'])
        if row.status != 'ok' and (faith != 0 or similarity != 0):
            raise ValueError(f'Cached baseline score contradicts saved status: {row.method} {row.dataset_id}')
    descriptive_rows.append({'dataset_id': row.dataset_id, 'method': row.method,
                             'faithfulness': faith, 'similarity': similarity})
descriptive = pd.DataFrame(descriptive_rows)
descriptive.to_csv(OUT / 'pangaea_descriptive_per_dataset.csv', index=False)


def technical_summary(corpus, method):
    selected = technical.loc[(technical.corpus == corpus) & (technical.method == method)]
    out = {}
    for field in TECH_FIELDS:
        part = selected.loc[selected.field == field]
        correct = int(part.correct.sum())
        out[f'{field}_accuracy'] = correct / N_PER_CORPUS
        if corpus == 'pangaea':
            tp = int((part.reference_present & part.prediction_present).sum())
            fp = int((~part.reference_present & part.prediction_present).sum())
            fn = int((part.reference_present & ~part.prediction_present).sum())
            out[f'{field}_precision'] = tp/(tp+fp) if tp+fp else np.nan
            out[f'{field}_recall'] = tp/(tp+fn) if tp+fn else np.nan
    out['technical_accuracy'] = float(np.mean([out[f'{f}_accuracy'] for f in TECH_FIELDS]))
    out['spatial_accuracy'] = float(np.mean([out[f'{f}_accuracy'] for f in SPATIAL]))
    out['temporal_accuracy'] = float(np.mean([out[f'{f}_accuracy'] for f in TEMPORAL]))
    if corpus == 'pangaea':
        for metric in ('precision', 'recall'):
            scores = [out[f'{f}_{metric}'] for f in TECH_FIELDS]
            out[f'technical_{metric}'] = float(np.nanmean(scores)) if not np.isnan(scores).all() else np.nan
    return out


summary_rows = []
for corpus in ('pangaea', 'inaturalist'):
    for method in ('M3_fast', 'ReAct', 'SingleLLM'):
        selected = predictions.loc[(predictions.corpus == corpus) & (predictions.method == method)]
        if len(selected) != N_PER_CORPUS:
            raise ValueError(f'Incomplete selection: {corpus} {method}')
        counts = selected.status.value_counts()
        scores = technical_summary(corpus, method)
        row = {'corpus': corpus, 'method': method, 'n': N_PER_CORPUS,
               'successes': int(counts.get('ok', 0)),
               'skips': int(counts.get('skipped', 0)),
               'failures_or_missing': int(N_PER_CORPUS-counts.get('ok', 0)-counts.get('skipped', 0)),
               **scores}
        if corpus == 'pangaea':
            parts = descriptive.loc[descriptive.method == method]
            row['faithfulness'] = float(parts.faithfulness.mean())
            row['similarity'] = float(parts.similarity.mean())
        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary.to_csv(OUT / 'comparison_summary.csv', index=False)
print('PANGAEA (percent of 20-file end-to-end sample):')
print((summary.loc[summary.corpus == 'pangaea',
                   ['method', 'successes', 'skips', 'failures_or_missing',
                    'faithfulness', 'similarity', 'technical_accuracy',
                    'technical_precision', 'technical_recall']]
       .assign(**{name: lambda d, name=name: (100*d[name]).round(1)
                  for name in ('faithfulness', 'similarity', 'technical_accuracy',
                               'technical_precision', 'technical_recall')})
       .to_string(index=False)))
print('\niNaturalist (exact technical accuracy; percent of 20-file sample):')
print((summary.loc[summary.corpus == 'inaturalist',
                   ['method', 'successes', 'skips', 'failures_or_missing',
                    'technical_accuracy', 'spatial_accuracy', 'temporal_accuracy']]
       .assign(**{name: lambda d, name=name: (100*d[name]).round(1)
                  for name in ('technical_accuracy', 'spatial_accuracy', 'temporal_accuracy')})
       .to_string(index=False)))
print('Saved:', OUT / 'comparison_summary.csv')


PANGAEA (percent of 20-file end-to-end sample):
   method  successes  skips  failures_or_missing  faithfulness  similarity  technical_accuracy  technical_precision  technical_recall
  M3_fast         19      0                    1          90.0        19.0                64.2                100.0              88.0
    ReAct         20      0                    0         100.0        34.2                64.2                 97.8              91.9
SingleLLM         19      1                    0          78.8        37.6                59.2                 95.6              85.9

iNaturalist (exact technical accuracy; percent of 20-file sample):
   method  successes  skips  failures_or_missing  technical_accuracy  spatial_accuracy  temporal_accuracy
  M3_fast         20      0                    0               100.0             100.0              100.0
    ReAct         20      0                    0                95.0             100.0               85.0
SingleLLM          0     20   